## 1. Importações e Configurações de Parâmetros

In [0]:
import re
import time
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("schema_origem", "raw")
dbutils.widgets.text("volume_origem", "csvs")

catalogo       = dbutils.widgets.get("catalogo")
schema_origem  = dbutils.widgets.get("schema_origem")
volume_origem  = dbutils.widgets.get("volume_origem")

spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

volume_path = f"/Volumes/{catalogo}/{schema_origem}/{volume_origem}"

print(f"Catálogo em uso: {catalogo}")
print(f"Volume de origem: {volume_path}")

Catálogo em uso: workspace
Volume de origem: /Volumes/workspace/raw/csvs


## 2. Mapeamento de fontes

In [0]:
fontes = {
    "tb_clientes":    "clientes.csv",
    "tb_pedidos":     "pedidos.csv",
    "tb_produtos":    "catalogo_produtos.csv",
    "tb_tickets":     "suporte_tickets.csv",
    "tb_clickstream": "clickstream.csv",
    "tb_avaliacoes":  "avaliacoes.csv",
}

print(f"Fontes mapeadas: {len(fontes)} arquivos")

Fontes mapeadas: 6 arquivos


## 3. Tabela de auditoria `bronze.dq_log`
Histórico persistente das execuções do pipeline. Registra tabela, etapa, volumetria, duração e origem por lote ingerido. A Silver consulta essa tabela para calcular taxa de descarte por regra de limpeza.

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS bronze.dq_log (
        execucao_id      STRING,
        camada           STRING,
        tabela           STRING,
        etapa            STRING,
        registros        BIGINT,
        duracao_segundos DOUBLE,
        arquivo_origem   STRING,
        timestamp_log    TIMESTAMP
    ) USING DELTA
""")

print("Tabela bronze.dq_log pronta")

Tabela bronze.dq_log pronta.


## 4. Função auxiliar de sanitização de coluna

Delta Lake não aceita caracteres especiais em nomes de coluna: espaço, acento, hífen e parênteses causam falha de escrita. A função substitui tudo fora de `[a-z0-9_]` por underscore e força minúsculo, normalizando o que vier do sistema de origem.

In [0]:
def sanitizar_colunas(df: DataFrame) -> DataFrame:
    def limpar(nome: str) -> str:
        return re.sub(r"[^\w]", "_", nome.strip()).lower()
    return df.toDF(*[limpar(c) for c in df.columns])

print("Função de sanitização definida")

Função de sanitização definida.


## 5. Função de ingestão Bronze

Padrão único aplicado às 6 tabelas:

**`inferSchema=False`:** A Bronze preserva o dado exatamente como veio da fonte. Quando a inferência automática está ativa, o Spark promove tipos silenciosamente e descarta linhas que não se encaixem na promoção, comportamento que viola o princípio da camada. A tipagem é responsabilidade da Silver, onde acontece de forma controlada e rastreável.

**`timestamp_ingestion`:** Gerado no momento da escrita Delta. Cumpre duas funções: registra quando cada lote entrou no lakehouse para auditoria, e serve de âncora para a deduplicação na Silver, que usa esse campo para resolver duplicatas por chave de negócio mantendo o registro mais recente.

**`arquivo_origem` com path absoluto:** Grava o caminho completo do Volume em vez do nome do arquivo. Garante rastreabilidade quando houver re-ingestão a partir de outro Volume, ambiente ou bucket de origem, cenário comum em pipelines que evoluem ao longo do tempo.

**`overwriteSchema=true`:** Permite evolução de schema entre versões do dataset sem quebrar o pipeline com erro de incompatibilidade. Decisão consciente de tolerância a mudança na origem, alinhada com o papel da Bronze de absorver o que vier.

**Contagem via `spark.table` pós-escrita:** Chamar `df.count()` antes da escrita força um re-scan completo do CSV, dobrando o tempo de processamento em tabelas grandes. Após a materialização, a tabela Delta resolve a contagem usando as estatísticas registradas no transaction log, sem reprocessar dados.

In [0]:
def ingerir_bronze(arquivo: str, tabela: str, execucao_id: str) -> tuple[int, float]:
    inicio = time.time()
    caminho = f"{volume_path}/{arquivo}"

    df = (
        spark.read
        .option("header",      "true")
        .option("inferSchema", "false")
        .option("multiLine",   "true")
        .option("escape",      '"')
        .option("encoding",    "UTF-8")
        .csv(caminho)
    )

    df = (
        sanitizar_colunas(df)
        .withColumn("timestamp_ingestion", F.current_timestamp())
        .withColumn("arquivo_origem",      F.lit(caminho))
    )

    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"bronze.{tabela}")
    )

    registros = spark.table(f"bronze.{tabela}").count()
    duracao = time.time() - inicio

    log_entry = spark.createDataFrame(
        [(execucao_id, "bronze", tabela, "ingestao", registros, duracao, caminho, datetime.now())],
        ["execucao_id", "camada", "tabela", "etapa", "registros", "duracao_segundos", "arquivo_origem", "timestamp_log"]
    )
    log_entry.write.format("delta").mode("append").saveAsTable("bronze.dq_log")

    return registros, duracao

print("Função de ingestão definida")

Função de ingestão definida.


## 6. Execução da ingestão

In [0]:
execucao_id = datetime.now().strftime("%Y%m%d_%H%M%S")
resultados = {}

for tabela, arquivo in fontes.items():
    qtd, duracao = ingerir_bronze(arquivo, tabela, execucao_id)
    resultados[tabela] = qtd
    print(f"bronze.{tabela:20s}  {qtd:>9,} registros  {duracao:>6.2f}s")

print(f"\nTotal ingerido: {sum(resultados.values()):,} registros em {len(resultados)} tabelas")
print(f"Execução registrada em bronze.dq_log com id: {execucao_id}")


bronze.tb_clientes              61,345 registros    3.80s
bronze.tb_pedidos              314,900 registros    4.21s
bronze.tb_produtos                 517 registros    3.10s
bronze.tb_tickets               34,697 registros    3.07s
bronze.tb_clickstream          500,000 registros    4.88s
bronze.tb_avaliacoes           156,832 registros    3.37s

Total ingerido: 1,068,291 registros em 6 tabelas
Execução registrada em bronze.dq_log com id: 20260503_070512


## 7. Validação técnica da camada Bronze

Quality gate executado ao final do notebook. Em caso de falha, lança exceção para interromper o Workflow antes que a Silver consuma dados ausentes ou inconsistentes. Quatro regras compõem a validação.

1. **Existência.** As 6 tabelas mapeadas devem estar presentes no catálogo. Cobre o caso de uma escrita silenciosamente abortada por erro de permissão ou rede.
2. **Volumetria.** Nenhuma tabela pode estar vazia. Captura o caso em que o CSV de origem foi substituído por um arquivo válido mas sem registros, situação que passaria por todas as outras validações.
3. **Schema mínimo.** Cada tabela precisa de pelo menos uma coluna de dado além das duas de auditoria. Protege contra CSV corrompido com header presente mas sem conteúdo, caso de borda mas que passaria nas validações de existência e volumetria.
4. **Auditoria.** As colunas `timestamp_ingestion` e `arquivo_origem` devem existir em todas as tabelas. Esse é o contrato consumido pela Silver para deduplicação e rastreio.

In [0]:
print("\nValidação final da camada Bronze")

falhas = []

for tabela in fontes.keys():
    nome_completo = f"bronze.{tabela}"

    if not spark.catalog.tableExists(nome_completo):
        print(f"[ERRO] {nome_completo} não encontrada")
        falhas.append(f"{nome_completo} ausente")
        continue

    df = spark.table(nome_completo)
    total = df.count()

    if total == 0:
        print(f"[ERRO] {nome_completo} vazia")
        falhas.append(f"{nome_completo} vazia")
        continue

    colunas_obrigatorias = {"timestamp_ingestion", "arquivo_origem"}
    colunas_ausentes = colunas_obrigatorias - set(df.columns)

    if colunas_ausentes:
        print(f"[ERRO] {nome_completo} sem colunas de auditoria: {colunas_ausentes}")
        falhas.append(f"{nome_completo} sem {colunas_ausentes}")
        continue

    colunas_dados = set(df.columns) - colunas_obrigatorias
    if len(colunas_dados) == 0:
        print(f"[ERRO] {nome_completo} sem colunas de dados (apenas auditoria)")
        falhas.append(f"{nome_completo} sem colunas de dados")
        continue

    print(f"[OK]   {nome_completo}: {total:,} registros, {len(colunas_dados)} colunas de dados")

if falhas:
    raise Exception("Validação final da Bronze falhou:\n- " + "\n- ".join(falhas))

print("\n[SUCESSO] Camada Bronze validada")


Validação final da camada Bronze
[OK]   bronze.tb_clientes: 61,345 registros, 14 colunas de dados
[OK]   bronze.tb_pedidos: 314,900 registros, 8 colunas de dados
[OK]   bronze.tb_produtos: 517 registros, 10 colunas de dados
[OK]   bronze.tb_tickets: 34,697 registros, 9 colunas de dados
[OK]   bronze.tb_clickstream: 500,000 registros, 11 colunas de dados
[OK]   bronze.tb_avaliacoes: 156,832 registros, 9 colunas de dados

[SUCESSO] Camada Bronze validada.
